<a href="https://colab.research.google.com/github/Umair911/Final/blob/main/penglab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Penglab (Abuse of Google Colab for fun and profit)
#by mxrch

#Choose what you want to install
hashcat = True
john = False
hydra = False

#Which shell ? (only one, or none)
#⚠️ Don't select a shell to run bash commands in the Google Colab way (you'll see). Stable.
ssh = False #It uses ngrok. Stable and interactive, but take a little time to setup
python_shell = False #Useful for little hashes, but not for long hours, since you'll see the output only when it finish. Not interactive, but stable.
bash_shell = True #Useful for big hashes. but you'll don't see your input (surely a Google Colab protection). Stable.

#Wordlists (see weakpass.com to download them)
wordlists_dir = "wordlists"

rockyou = True #For CTFs, especially Hack The Box (133.44 Mb)
hashesorg2019 = False #Very heavy (12.79 Gb) but has a good rate, if you're determinated

#---------------------------

if (python_shell and bash_shell) or (bash_shell and ssh) or (ssh and python_shell) :
    print("Please do a choice")
    exit()

In [6]:
#Wordlists
import os

os.system("wordlists_dir={}".format(wordlists_dir))
!mkdir ./$wordlists_dir
if rockyou:
    !printf "[+] Downloading the Rockyou wordlist...\n"
    !cd $wordlists_dir && wget https://download.weakpass.com/wordlists/90/rockyou.txt.gz
    !printf "[+] Wordlist downloaded !\n[+] Extraction...\n"
    !cd $wordlists_dir && gunzip rockyou.txt.gz
    !printf "[+] Finished !\n[+] Location : $(pwd)/$wordlists_dir/$(ls wordlists | grep rockyou)"

if hashesorg2019:
    !printf "[+] Downloading the HashesOrg2019 wordlist...\n"
    !cd $wordlists_dir && wget https://download.weakpass.com/wordlists/1851/hashesorg2019.gz
    !printf "[+] Wordlist downloaded !\n[+] Extraction...\n"
    !cd $wordlists_dir && gunzip hashesorg2019.gz
    !printf "[+] Finished !\n[+] Location : $(pwd)/$wordlists_dir/$(ls wordlists | grep hashesorg2019)"

[+] Downloading the Rockyou wordlist...
--2026-09-09 18:57:07--  https://download.weakpass.com/wordlists/90/rockyou.txt.gz
Resolving download.weakpass.com (download.weakpass.com)... 104.21.22.190, 172.67.206.163, 2606:4700:3030::ac43:cea3, ...
Connecting to download.weakpass.com (download.weakpass.com)|104.21.22.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53357062 (51M) [application/octet-stream]
Saving to: ‘rockyou.txt.gz’

rockyou.txt.gz      100%[===================>]  50.88M  45.1MB/s    in 1.1s    

2026-09-09 18:57:09 (45.1 MB/s) - ‘rockyou.txt.gz’ saved [53357062/53357062]

[+] Wordlist downloaded !
[+] Extraction...
[+] Finished !
[+] Location : /content/wordlists/rockyou.txt

In [7]:
#Install of hashcat
if hashcat:
    print("Installation of hashcat...")
    !apt install cmake build-essential -y
    !apt install checkinstall git -y
    !git clone https://github.com/hashcat/hashcat.git && cd hashcat && make -j 8 && make install

Installation of hashcat...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.28.3-1build7).
build-essential is already the newest version (12.10ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.43.0-1ubuntu7.3).
Suggested packages:
  gettext
The following NEW packages will be installed:
  checkinstall
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 106 kB of archives.
After this operation, 442 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 checkinstall amd64 1.6.2+git20170426.d24a630-4 [106 kB]
Fetched 106 kB in 1s (106 kB/s)
Selecting previously unselected package checkinstall.
(Reading database ... 126952 files and directories currently installed.)
Preparing to

In [ ]:
#Install of john
if john:
    print("Installation of john...")
    !apt-get install john -y

In [ ]:
#Install of hydra
if hydra:
    print("Installation of hydra...")
    !apt install cmake build-essential -y
    !apt install checkinstall git -y
    !git clone https://github.com/vanhauser-thc/thc-hydra.git && cd thc-hydra && ./configure && make && make install

In [1]:
#Setting up shells
from termcolor import colored
import os

if ssh:
    print("Setting up SSH...")
    !pip install git+https://github.com/demotomohiro/remocolab.git
    import remocolab
    remocolab.setupSSHD()

elif python_shell:
    os.system("touch /tmp/cmd && touch /tmp/status")

    template_cmd = "echo -n $(whoami)[$(hostname)[$(pwd) &> /tmp/status"
    os.system("bash -c '{}'".format(template_cmd))
    output = {"cmd": "", "status": ""}
    with open('/tmp/cmd', 'r') as f:
        output["cmd"] = f.read()
    with open('/tmp/status', 'r') as f:
        output["status"] = f.read()
    prefixes = output["status"].split("[")
    path = prefixes[2].replace('\n', '')
    prefix = colored(prefixes[0] + "@" + prefixes[1], "red") + ":" + colored(path, "cyan") + "$ "
    print("")

    while 1:
        print(prefix, end='')
        cmd = input()
        template_cmd = "cd {} && ".format(path) + "" + cmd + " &> /tmp/cmd ; echo $(whoami)[$(hostname)[$(pwd) &> /tmp/status"
        os.system("bash -c '{}'".format(template_cmd))
        output = {"cmd": "", "status": ""}
        with open('/tmp/cmd', 'r') as f:
            output["cmd"] = f.read()
        with open('/tmp/status', 'r') as f:
            output["status"] = f.read()
        prefixes = output["status"].split("[")
        path = prefixes[2].replace('\n', '')
        prefix = colored(prefixes[0] + "@" + prefixes[1], "red") + ":" + colored(path, "cyan") + "$ "
        print(output["cmd"])

elif bash_shell:
    !/bin/bash

else:
    print('\n🚨🚨🚨\n🎉 Your environment is ready !\nWant to run it from the code blocks ?')
    print('Just use the code block below and start your commands by a "!" (make sure to uncomment them).')
    print('▶️ When you\'re ready, hit the run button at the left of the code block !\n')
    print('|\n|\n|\nV')

NameError: name 'ssh' is not defined

In [ ]:
# Put your code here :
# !hashcat -h
# !john --help